### Crypto-ML-Pairs-Trading (CLEAN) (Perp. Futures) (BinanceEX/HyperLiquid)

<b><u> Statistical-Arbitraging </u></b>

1. Start beta-neutral to isolate and test if strategy generates true alpha, independent of market moves.

2. Evaluate signal quality, if consistent alpha shows up, it's a valid edge and not just hidden beta.

3. Add back beta exposure (e.g., via index ETFs) to scale returns, creating a robust (beta + alpha) portfolio.

<p align="center">
  <span style="font-size:24px; font-weight:bold;"> Trade System Architecture (R&D) </span><br>
  <img src="../theory/ML-pairs-open-arch.png" width="40%">
</p>


#### Research

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import math
import os
from pprint import pprint
import seaborn as sns
from IPython.display import display
import time
from warnings import filterwarnings
import importlib
import pickle
import json

filterwarnings('ignore') 

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering

In [ ]:
import utils
import utils.methods
import utils.retrieve
import utils.cluster
import utils.benchmark
import utils.portfolio
import utils.position

importlib.reload(utils.methods)
importlib.reload(utils.retrieve)
importlib.reload(utils.cluster)
importlib.reload(utils.benchmark)
importlib.reload(utils.portfolio)
importlib.reload(utils.position)

In [ ]:
from utils.methods import *
from utils.retrieve import *
from utils.cluster import *
from utils.benchmark import *
from utils.portfolio import *
from utils.position import *

In [ ]:
with open("config.json", "r") as f:
    config = json.load(f)

# Access variables
const_inner_col = config["cryp_tmp_fields"]
cryp_market = config["cryp_benchmark"]
start = config["start"]
end = config["end"]
day_limit = config["day_limit"]
variance_perc = config["variance_perc"]
cluster_distance_threshold = config["cluster_distance_threshold"]
cluster_percentile = config["cluster_percentile"]

print(const_inner_col, start, end)

Data-Load (BinanceEX API)

In [ ]:
stable_ticker = 'USDC/USDT'
market_ticker = 'BTC/USDT'

In [ ]:
df_cryp_uni, cryp_uni_list = get_ccxt_crypto_data(timeframe='1d', limit=1000, is_plot=False, load_universe=True)

market_df = df_cryp_uni.loc[market_ticker]
stablecoin_df = df_cryp_uni.loc[stable_ticker]

df_cryp_uni.drop([market_ticker, stable_ticker], axis=0)
cryp_uni_list = [c for c in cryp_uni_list if c not in [stable_ticker, market_ticker]]

In [ ]:
print('Selected-coins (except market & stable): ', len(cryp_uni_list), cryp_uni_list[:5])
print()
display(df_cryp_uni.head(2))
print()
display(market_df.head(2))
print()
display(stablecoin_df.head(2))

#### ~2.5 Years API Call-Limit

In [ ]:
rn_coin = random.randint(0, len(cryp_uni_list))
rn_coin = cryp_uni_list[rn_coin]
print('Rnd-Coin-Chosen: ', rn_coin)

print('-- Sampled-Date-Range --')
print(f'start: {market_df.index[0]}, end: {market_df.index[-1]}')

fig, axs = plt.subplots(1, 3, figsize=(15, 4))

market_df.close.plot(ax=axs[0])
axs[0].set_title('Market-Rates (BTC/USDT)')

stablecoin_df.close.plot(ax=axs[1])
axs[1].set_title('StableCoin-Rates (USDC/USDT)')
axs[1].set_ylim(0.995, 1.005)

df_cryp_uni.loc[rn_coin].close.plot(ax=axs[2])
axs[2].set_title(f'Coin-Selected: {rn_coin}')

plt.tight_layout(pad=2)
plt.show()

In [ ]:
sub_sample = 20

In [ ]:
stocks = np.random.choice(np.array(cryp_uni_list), size=sub_sample).tolist()
print('sampled-selected: ', stocks)

df_uni = df_cryp_uni.copy()
df_uni = df_uni.loc[stocks]

df_cleaned = []
for nm in stocks:
    df_filtered = df_uni.loc[nm].copy()
    df_filtered = df_filtered[~df_filtered.index.duplicated(keep='first')]
    df_filtered['symbol'] = nm  
    df_cleaned.append(df_filtered)

# Combine and set multi-index again
df_uni = pd.concat(df_cleaned).set_index('symbol', append=True).swaplevel()
df_uni = df_uni.sort_index()

display(df_uni.head(3))

Multi-Visualization (RAW)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(18, 5))

### We just do multi-indexing
for nm in stocks:
    axs[0].plot(df_uni.loc[nm].index, df_uni.loc[nm]['close'], label = nm)
    axs[0].set_xlabel('Date')
    axs[0].set_ylabel('Close')

plt.legend()

for nm in stocks:

    pct_change_clean = df_uni.loc[nm]['close'].pct_change().dropna()

    if pct_change_clean.empty:
        print(f"Skipping {nm} due to no valid percentage change data.")
        continue
    
    axs[1].hist(pct_change_clean, bins = 50, label = nm)
    axs[1].set_title(f'PCT Change Distribution')

plt.tight_layout(pad = 2)
plt.legend()
plt.show()

Data-Cleaning 

In [ ]:
history_limit = 1000

## returns factor-arrays (takes types and fundamental_df)
val_arr = data_clean(df_uni=df_uni, stocks=stocks, history_limit=history_limit, type='cryp', fund_df=None, is_plot=True)
    
print(len(val_arr), len(val_arr[0]))
val_arr = np.array(val_arr)
print(val_arr.shape)

print('New-Stocks: ', stocks)
print(len(stocks))

val_arr = val_arr.reshape(len(stocks), -1)
print(val_arr.shape)

PortFolio-Construction

In [ ]:
import utils
import utils.retrieve
import utils.benchmark
import utils.portfolio

importlib.reload(utils.retrieve)
importlib.reload(utils.benchmark)
importlib.reload(utils.portfolio)

from utils.retrieve import *
from utils.benchmark import *
from utils.portfolio import *

In [ ]:
from utils.factor_model import *
from utils.alpha_ex import *
from utils.beta_ex import *
from utils.portfolio import *
from utils.risk_engine import *
from utils.backtest import *
from utils.forward_test import *
from utils.analytics import *

##### For the research env everything will be executed independently akin to their modularized nature

Predictive-Arbitrage (Factor-Model -> Risk-Engine)

In [ ]:
## Call factor_model and pass the retrieved clean/data
factor_model = FactorModelPairs(stocks, variance_perc, is_plot=False)
pairs = factor_model.run()

In [ ]:
## Call alpha_ex and pass the coint_pairs
alpha = PairsTradingAlpha(pairs, lookback_window, entry_z, exit_z)
target_position_alpha = alpha.generate_signal(timestamp, current_positions)

In [ ]:
## Call beta_ex and get the beta_exposure
beta = SimpleMarketBeta(market_proxy=market_df, long_term_ma=60)
target_position_beta = beta.generate_signal(timestamp)

In [ ]:
## Call portfolio and pass the alpha and beta exposure
portfolio = PortfolioConstructor(alpha_allocation, beta_allocation, beta_basket)
portfolio_weights = portfolio.construct_target_weights(alpha_signals, beta_signal)

In [ ]:
## Call risk_engine and add execution/liquidity/volume/positional caps/constraints
risk_manager = RiskManager(max_leverage, max_concentration, max_portfolio_drawdown)
hedged_portfoilo_weights = risk_manager.manage_weights(target_weights, portfolio_state)

Portfoilio-Backtesting

In [ ]:
## Call backtest and pass the risk-hedged position to test on out-of-sample historical data 
backtester = PerpetualFuturesBacktester(initial_capital, fee)
equity_series = backtester.run(data, funding_data, orchestrator: callable)

In [ ]:
display(equity_series)

In [ ]:
## Call forward test and pass the risk-hedged position for forward/live testing


In [ ]:
## Call/Save analytics on both the tests 